# 08 — Long-Form Probability Estimation: GT vs Tracker

Extends notebook 08 from single clips to a **full game** (e.g. AY_G1).

A full game is composed of many short chunks (`AY_G1_chunk_0000`, `0001`, …).
This notebook:

1. **Groups** all chunks belonging to one game by name prefix
2. **Concatenates** their MOT ground-truth annotations into a single timeline,
   re-numbering frames so track IDs are globally consistent across chunks
3. Runs the YOLO11 detector + tuned BoTSORT on every chunk video in order,
   applying the same `merge_fragments()` post-processor from notebook 07 to
   stitch tracks across chunk boundaries
4. Accumulates per-frame cumulative stat counts for both GT and tracker
5. Computes and plots the evolving posterior P(minigame | counts) for both,
   side-by-side across the full game timeline

Prerequisites: `02_train_yolo.ipynb` · `07_tracker_tuning.ipynb` · `08_probability_distributions.ipynb`

## 0 — Configuration

In [ ]:
from pathlib import Path
import json, cv2, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from ultralytics import YOLO

REPO_ROOT    = Path('/home1/hendersonj6179@cgu.edu')
RUNS_DIR     = (REPO_ROOT / 'runs').resolve()
MOT_SEQ_DIR  = REPO_ROOT / 'data/mot_dataset/sequences'
CHUNKS_DIR   = REPO_ROOT / 'data/processed/labelstudiochunks'
OUT_DIR      = (REPO_ROOT / 'runs/longform').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Select the full game to analyse ──────────────────────────────────────
# Set to the shared prefix of all chunks belonging to one game.
# All sequences whose name starts with this prefix will be used in order.
GAME_PREFIX = 'AY_G1'    # e.g. 'AY_G1', 'SerpentBoi_G2', 'Frozoha_G1', ...

# ── Detector weights ──────────────────────────────────────────────────────
WEIGHTS     = RUNS_DIR / 'yolo26' / 'final' / 'weights' / 'best.pt'
# WEIGHTS   = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'

# ── Tracker config: baseline arch config from notebook 04 ────────────────
# HP tuning (notebooks 05/05b) was abandoned — baseline outperforms tuned configs.
_ARCH_CFG_PATH = REPO_ROOT / 'configs' / 'kartector_botsort_arch.json'
if _ARCH_CFG_PATH.exists():
    import json as _jcfg08
    _arch_cfg08   = _jcfg08.loads(_ARCH_CFG_PATH.read_text())
    TRACKER_CFG   = str(REPO_ROOT / _arch_cfg08.get('tracker_yaml', 'configs/kartector_botsort_reentry.yaml'))
    CONF          = _arch_cfg08.get('conf', 0.25)
    IOU           = 0.70
    REID_OPTION   = _arch_cfg08.get('reid_option', 'B')
    REID_B_PARAMS = _arch_cfg08.get('reid_b_params', {})
else:
    TRACKER_CFG   = str(REPO_ROOT / 'configs' / 'kartector_botsort_reentry.yaml')
    CONF = 0.25; IOU = 0.70
    REID_OPTION   = 'B'
    REID_B_PARAMS = {}
if not Path(TRACKER_CFG).exists():
    TRACKER_CFG = 'botsort.yaml'

def _make_boxmot_tracker08():
    """Instantiate a fresh BotSort tracker using the baseline arch config."""
    import torch
    from boxmot.trackers.botsort.botsort import BotSort
    from boxmot.utils import WEIGHTS as _BW
    device = torch.device(
        'cuda' if torch.cuda.is_available() else
        'mps'  if torch.backends.mps.is_available() else 'cpu')
    return BotSort(
        reid_weights=_BW / 'osnet_x0_25_msmt17.pt',
        device=device, half=False,
        track_buffer=REID_B_PARAMS.get('track_buffer', 30),
        match_thresh=REID_B_PARAMS.get('match_thresh', 0.70),
        appearance_thresh=REID_B_PARAMS.get('appearance_thresh', 0.40),
        proximity_thresh=REID_B_PARAMS.get('proximity_thresh', 0.5),
        with_reid=True,
    )

with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES   = splits['classes']
N_CLASSES = len(CLASSES)

# Discover all sequences for this game, in chunk order
all_seqs = sorted(MOT_SEQ_DIR.iterdir())
game_seqs = [s for s in all_seqs if s.name.startswith(GAME_PREFIX) and s.is_dir()]
print(f'Game prefix : {GAME_PREFIX}')
print(f'Chunks found: {len(game_seqs)}')
print(f'Weights     : {WEIGHTS}')
print(f'Tracker     : {TRACKER_CFG}')

## 1 — Minigame Model

Paste the real distributions from notebook 08 here once confirmed.
> **TODO:** Replace placeholder uniform distributions.

In [ ]:
MINIGAMES = [
    'Single Race',
    'Drag Race',
    'Destruction Derby',
]
PRIOR = np.ones(len(MINIGAMES)) / len(MINIGAMES)
LIKELIHOODS = np.array([
    # Boost  Charge  Defense  Glide   HP      Offense  TopSpeed  Turn    Weight
    [0.083,0.083,  0.083,   0.083,  0.075,  0.062,   0.083,    0.083,  0.083],  # Single Race
    [0.082,0.082,  0.082,   0.082,  0.074,  0.082,   0.082,    0.082,  0.074],  # Drag Race
    [0.09, 0.072,  0.108,   0.072,  0.072,  0.129,    0.05,    0.072,  0.072],  # Destruction Derby
])
# Normalise each row so it sums to exactly 1
LIKELIHOODS = LIKELIHOODS / LIKELIHOODS.sum(axis=1, keepdims=True)

def compute_posterior(counts, prior=PRIOR, likelihoods=LIKELIHOODS):
    log_p = np.log(prior + 1e-12) + np.dot(np.log(likelihoods + 1e-12), counts)
    log_p -= log_p.max()
    p = np.exp(log_p); return p / p.sum()

print('Minigame model ready.')

## 2 — Build Ground-Truth Long-Form Timeline

Reads `gt.txt` from each chunk’s MOT sequence folder and concatenates them into
a single DataFrame, offsetting frame numbers so the timeline is continuous.

GT format (MOT17): `frame, track_id, x, y, w, h, conf, class_id, visibility`
Frames are 1-indexed in the files; we convert to 0-indexed here.

Track IDs are made globally unique across chunks by adding a large per-chunk offset
so that track 1 from chunk 0002 is never confused with track 1 from chunk 0003.

In [ ]:
def load_gt_chunk(seq_dir):
    gt_file = seq_dir / 'gt' / 'gt.txt'
    rows = []
    if not gt_file.exists():
        return pd.DataFrame(columns=['frame','track_id','class_id','x1','y1','x2','y2'])
    for line in gt_file.read_text().splitlines():
        parts = line.strip().split(',')
        if len(parts) < 8: continue
        frame_id = int(parts[0]) - 1          # 0-indexed
        track_id = int(parts[1])
        x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
        class_id  = int(parts[7]) - 1         # MOT gt.txt is 1-based; convert to 0-based
        rows.append({'frame': frame_id, 'track_id': track_id, 'class_id': class_id,
                     'x1': x, 'y1': y, 'x2': x + w, 'y2': y + h})
    return pd.DataFrame(rows)


def get_seq_length(seq_dir):
    ini = seq_dir / 'seqinfo.ini'
    if ini.exists():
        for line in ini.read_text().splitlines():
            if line.startswith('seqLength'):
                return int(line.split('=')[1])
    # fallback: count images
    return len(list((seq_dir / 'img1').glob('*.jpg')))


gt_rows_all = []
frame_offset = 0
tid_offset   = 0
chunk_boundaries = []   # (chunk_name, global_start_frame, global_end_frame)
TID_STRIDE = 100_000    # large enough to prevent any cross-chunk ID collision

for seq in tqdm(game_seqs, desc='Loading GT'):
    chunk_df = load_gt_chunk(seq)
    seq_len  = get_seq_length(seq)
    if not chunk_df.empty:
        chunk_df['frame']    += frame_offset
        chunk_df['track_id'] += tid_offset
        gt_rows_all.append(chunk_df)
    chunk_boundaries.append((seq.name, frame_offset, frame_offset + seq_len - 1))
    frame_offset += seq_len
    tid_offset   += TID_STRIDE

gt_df = pd.concat(gt_rows_all, ignore_index=True) if gt_rows_all else pd.DataFrame(
    columns=['frame','track_id','class_id','x1','y1','x2','y2'])
total_frames = frame_offset

# Apply merge_fragments to GT to stitch tracks broken at chunk boundaries.
# Within each chunk tracks are already correct; only cross-boundary resets need repair.
gt_df_raw = gt_df.copy()
gt_df      = merge_fragments(gt_df, max_gap=60, max_dist_pct=35.0)
_merged_gt = gt_df_raw['track_id'].nunique() - gt_df['track_id'].nunique()
print(f'GT track IDs before merge : {gt_df_raw["track_id"].nunique()}')
print(f'GT track IDs after  merge : {gt_df["track_id"].nunique()}  ({_merged_gt} stitched)')


print(f'Total frames (GT timeline): {total_frames}')
print(f'GT detections             : {len(gt_df)}')
print(f'GT unique track IDs       : {gt_df["track_id"].nunique()}')
print(f'Chunks with annotations   : {sum(1 for c in game_seqs if (c/"gt"/"gt.txt").exists())} / {len(game_seqs)}')

## 3 — Run Tracker on All Chunks

Runs the detector + tracker on every chunk video in order, collecting per-frame
track results and then applying `merge_fragments()` from notebook 07 to stitch
track IDs that were broken when the tracker reset between chunks.

**Cross-chunk stitching logic:**
- Track IDs are made globally unique (same `TID_STRIDE` offset as GT)
- `merge_fragments()` is run once on the full concatenated result to bridge gaps
  that happen to fall exactly on chunk boundaries

In [ ]:
def merge_fragments(df, max_gap=45, max_dist_pct=35.0, frame_w=960, frame_h=540):
    if df.empty: return df.copy()
    diag = (frame_w**2 + frame_h**2) ** 0.5
    def centroid(rows):
        return (((rows['x1']+rows['x2'])/2).mean(), ((rows['y1']+rows['y2'])/2).mean())
    summaries = {}
    for tid, grp in df.groupby('track_id'):
        g = grp.sort_values('frame')
        summaries[tid] = {
            'class_id': int(g['class_id'].mode()[0]),
            'start': int(g['frame'].min()), 'end': int(g['frame'].max()),
            'exit_xy': centroid(g.tail(3)), 'entry_xy': centroid(g.head(3)),
        }
    parent = {t: t for t in summaries}
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b): parent[find(a)] = find(b)
    tids = sorted(summaries, key=lambda t: summaries[t]['start'])
    for i, ta in enumerate(tids):
        sa = summaries[ta]; candidates = []
        for tb_id in tids[i+1:]:
            sb = summaries[tb_id]
            if sb['start'] > sa['end'] + max_gap: break
            if sa['class_id'] != sb['class_id'] or sb['start'] - sa['end'] < 0: continue
            dx = sa['exit_xy'][0] - sb['entry_xy'][0]
            dy = sa['exit_xy'][1] - sb['entry_xy'][1]
            if ((dx**2+dy**2)**0.5)/diag*100 <= max_dist_pct:
                candidates.append(((dx**2+dy**2)**0.5, tb_id))
        if candidates: union(ta, min(candidates)[1])
    id_map = {t: find(t) for t in summaries}
    df2 = df.copy(); df2['track_id'] = df2['track_id'].map(id_map)
    return df2


# ── Define the 4 model × tracker combinations ────────────────────────────
# Each entry: (run_label, detector_weights, reid_option, reid_b_params,
#              tracker_cfg_yaml, use_bytetrack)
# botsort uses boxmot when REID_OPTION='B'; bytetrack always uses Ultralytics.

RTDETR_WEIGHTS = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'
BYTETRACK_CFG  = str(REPO_ROOT / 'configs' / 'kartector_bytetrack.yaml')
if not Path(BYTETRACK_CFG).exists():
    BYTETRACK_CFG = 'bytetrack.yaml'

RUNS_TO_EVAL = [
    {
        'label':       'YOLO + BoTSORT',
        'weights':     WEIGHTS,
        'tracker':     'botsort',
        'tracker_cfg': TRACKER_CFG,
        'reid_option': REID_OPTION,
        'reid_b_params': REID_B_PARAMS,
        'available':   WEIGHTS.exists(),
    },
    {
        'label':       'YOLO + ByteTrack',
        'weights':     WEIGHTS,
        'tracker':     'bytetrack',
        'tracker_cfg': BYTETRACK_CFG,
        'reid_option': 'none',
        'reid_b_params': {},
        'available':   WEIGHTS.exists(),
    },
    {
        'label':       'RT-DETR + BoTSORT',
        'weights':     RTDETR_WEIGHTS,
        'tracker':     'botsort',
        'tracker_cfg': TRACKER_CFG,
        'reid_option': REID_OPTION,
        'reid_b_params': REID_B_PARAMS,
        'available':   RTDETR_WEIGHTS.exists(),
    },
    {
        'label':       'RT-DETR + ByteTrack',
        'weights':     RTDETR_WEIGHTS,
        'tracker':     'bytetrack',
        'tracker_cfg': BYTETRACK_CFG,
        'reid_option': 'none',
        'reid_b_params': {},
        'available':   RTDETR_WEIGHTS.exists(),
    },
]

for _r in RUNS_TO_EVAL:
    status = '✓' if _r['available'] else '✗ weights not found'
    print(f"  {_r['label']:<25} {status}")


def _make_boxmot_tracker_cfg(reid_b_params):
    """Instantiate a fresh BotSort tracker from boxmot."""
    import torch
    from boxmot.trackers.botsort.botsort import BotSort
    from boxmot.utils import WEIGHTS as _BW
    device = torch.device(
        'cuda' if torch.cuda.is_available() else
        'mps'  if torch.backends.mps.is_available() else 'cpu')
    return BotSort(
        reid_weights=_BW / 'osnet_x0_25_msmt17.pt',
        device=device, half=False,
        track_buffer=reid_b_params.get('track_buffer', 30),
        match_thresh=reid_b_params.get('match_thresh', 0.70),
        appearance_thresh=reid_b_params.get('appearance_thresh', 0.40),
        proximity_thresh=reid_b_params.get('proximity_thresh', 0.5),
        with_reid=True,
    )


def run_tracker_longform(run_cfg, game_seqs, chunks_dir, conf, iou,
                         frame_offset_start=0, tid_offset_start=0):
    """
    Run one model×tracker combo over all chunks.
    Returns (trk_df_raw, elapsed_seconds).
    """
    import time
    import numpy as _npx
    from ultralytics import YOLO as _YOLO_LF

    det_model    = _YOLO_LF(str(run_cfg['weights']))
    reid_option  = run_cfg['reid_option']
    tracker_type = run_cfg['tracker']
    tracker_cfg  = run_cfg['tracker_cfg']

    trk_rows_all  = []
    frame_offset  = frame_offset_start
    tid_offset    = tid_offset_start
    t_start       = time.time()

    for seq in tqdm(game_seqs, desc=run_cfg['label'], leave=False):
        chunk_name = seq.name
        video_path = chunks_dir / f'{chunk_name}.mp4'
        seq_len    = get_seq_length(seq)
        if not video_path.exists():
            print(f'  [SKIP] {video_path.name}')
            frame_offset += seq_len; tid_offset += TID_STRIDE; continue

        rows = []
        if tracker_type == 'botsort' and reid_option == 'B':
            # ── boxmot BotSort ────────────────────────────────────────────
            _trk = _make_boxmot_tracker_cfg(run_cfg['reid_b_params'])
            _cap = cv2.VideoCapture(str(video_path))
            _fi  = 0
            while True:
                _ret, _frame = _cap.read()
                if not _ret: break
                _res  = det_model(_frame, conf=conf, verbose=False)
                _dets = _res[0].boxes
                if _dets is not None and len(_dets):
                    _darr = _npx.concatenate([
                        _dets.xyxy.cpu().numpy(),
                        _dets.conf.cpu().numpy().reshape(-1,1),
                        _dets.cls.cpu().numpy().reshape(-1,1)], axis=1)
                else:
                    _darr = _npx.empty((0, 6))
                for _t in _trk.update(_darr, _frame):
                    _cid = int(_t[6]) if len(_t) > 6 else 0
                    if _cid < 0 or _cid >= N_CLASSES: continue
                    rows.append({
                        'frame':    _fi + frame_offset,
                        'track_id': int(_t[4]) + tid_offset,
                        'class_id': _cid,
                        'x1': float(_t[0]), 'y1': float(_t[1]),
                        'x2': float(_t[2]), 'y2': float(_t[3]),
                        'conf': float(_t[5]),
                    })
                _fi += 1
            _cap.release()
        else:
            # ── Ultralytics tracker (botsort non-B or bytetrack) ──────────
            for _fi, r in enumerate(
                    det_model.track(str(video_path), tracker=tracker_cfg,
                                    conf=conf, iou=iou,
                                    stream=True, verbose=False, persist=False)):
                if r.boxes is not None and r.boxes.id is not None:
                    for box, tid, cls, c in zip(
                            r.boxes.xyxy.cpu().numpy(),
                            r.boxes.id.cpu().numpy().astype(int),
                            r.boxes.cls.cpu().numpy().astype(int),
                            r.boxes.conf.cpu().numpy()):
                        rows.append({
                            'frame':    _fi + frame_offset,
                            'track_id': int(tid) + tid_offset,
                            'class_id': int(cls),
                            'x1': box[0], 'y1': box[1],
                            'x2': box[2], 'y2': box[3],
                            'conf': float(c),
                        })

        frame_offset += seq_len; tid_offset += TID_STRIDE
        if rows:
            trk_rows_all.append(pd.DataFrame(rows))

    elapsed = time.time() - t_start
    _empty  = pd.DataFrame(columns=['frame','track_id','class_id','x1','y1','x2','y2','conf'])
    trk_raw = pd.concat(trk_rows_all, ignore_index=True) if trk_rows_all else _empty
    return trk_raw, elapsed


# ── Run all 4 combinations ─────────────────────────────────────────────────
all_run_results = {}   # label -> {trk_df, trk_df_raw, elapsed, posts, counts}

for run_cfg in RUNS_TO_EVAL:
    lbl = run_cfg['label']
    if not run_cfg['available']:
        print(f'\nSkipping {lbl} — weights not found.')
        continue
    print(f'\n{"─"*60}\n{lbl}\n{"─"*60}')
    trk_raw, elapsed = run_tracker_longform(
        run_cfg, game_seqs, CHUNKS_DIR, CONF, IOU)
    print(f'  Tracks before merge: {trk_raw["track_id"].nunique()}')
    trk_merged = merge_fragments(trk_raw, max_gap=60, max_dist_pct=35.0)
    print(f'  Tracks after  merge: {trk_merged["track_id"].nunique()}')
    print(f'  Elapsed            : {elapsed:.1f}s')
    trk_merged.to_csv(OUT_DIR / f'{GAME_PREFIX}_{lbl.replace(" ","_").replace("+","x")}_tracker.csv',
                      index=False)
    all_run_results[lbl] = {'trk_df': trk_merged, 'elapsed': elapsed}

print(f'\nCompleted {len(all_run_results)} / {len(RUNS_TO_EVAL)} runs.')


## 4 — Accumulate Posteriors Over the Full Timeline

For both GT and tracker, iterate frame-by-frame across the full game timeline,
counting each **new** track ID once (first time it appears) and updating the
Bayesian posterior.

In [ ]:
def accumulate_longform(df, n_classes, total_frames, prior, likelihoods):
    counts  = np.zeros(n_classes, dtype=int)
    seen    = set()
    posts   = np.zeros((total_frames, len(prior)))
    frame_map = {}
    for _, row in df.iterrows():
        f = int(row['frame'])
        frame_map.setdefault(f, []).append((int(row['track_id']), int(row['class_id'])))
    for f in range(total_frames):
        for tid, cid in frame_map.get(f, []):
            if tid not in seen:
                counts[cid] += 1
                seen.add(tid)
        posts[f] = compute_posterior(counts.copy(), prior, likelihoods)
    return posts, counts.copy()


# ── Accumulate posteriors for GT and every tracker run ───────────────────
print('Accumulating GT posteriors...')
gt_posts, gt_counts = accumulate_longform(
    gt_df, N_CLASSES, total_frames, PRIOR, LIKELIHOODS)

for lbl, res in all_run_results.items():
    posts, counts = accumulate_longform(
        res['trk_df'], N_CLASSES, total_frames, PRIOR, LIKELIHOODS)
    res['posts']  = posts
    res['counts'] = counts
    print(f'  {lbl}: posteriors accumulated')

print('Done.')


## 5 — Side-by-Side Posterior Evolution Plot

Plots the full-game posterior timeline for GT (top) and tracker (bottom).
Vertical dashed lines mark chunk boundaries so you can see exactly where
any tracking drift begins.

In [ ]:
# ── Plot: posterior evolution for each run vs GT ─────────────────────────
colors = plt.cm.Set1.colors
x = np.arange(total_frames)

for lbl, res in all_run_results.items():
    fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharey=True, sharex=True)
    for ax, posts, title in zip(axes,
                                 [gt_posts,       res['posts']],
                                 ['Ground Truth', lbl]):
        for i, mg in enumerate(MINIGAMES):
            ax.plot(x, posts[:, i], label=mg, lw=1.5, color=colors[i % len(colors)])
        for _, start, _ in chunk_boundaries:
            ax.axvline(start, color='gray', lw=0.5, alpha=0.4)
        ax.set_ylabel('P(minigame)'); ax.set_ylim(0, 1.05)
        ax.set_title(f'{title} — {GAME_PREFIX}', fontweight='bold')
        ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.2)
    axes[1].set_xlabel('Frame (global)')
    plt.suptitle(f'{lbl} vs GT — {GAME_PREFIX}', fontweight='bold', y=1.01)
    plt.tight_layout()
    _slug = lbl.replace(' ', '_').replace('+', 'x')
    _out  = OUT_DIR / f'{GAME_PREFIX}_{_slug}_posterior_evolution.png'
    plt.savefig(_out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {_out}')


## 6 — Overlay Comparison & Divergence

Overlays GT (solid) and tracker (dashed) on the same axes so divergence is
immediately visible, and plots the L1 difference between the two posteriors
over time.

In [ ]:
# ── Overlay + divergence for each run ────────────────────────────────────
for lbl, res in all_run_results.items():
    trk_posts = res['posts']
    fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharex=True,
                             gridspec_kw={'height_ratios': [3, 1]})
    for i, mg in enumerate(MINIGAMES):
        c = colors[i % len(colors)]
        axes[0].plot(x, gt_posts[:, i],  color=c, lw=1.5,       label=f'{mg} (GT)')
        axes[0].plot(x, trk_posts[:, i], color=c, lw=1.5, ls='--', alpha=0.7, label=f'{mg} (Trk)')
    for _, start, _ in chunk_boundaries:
        axes[0].axvline(start, color='gray', lw=0.5, alpha=0.4)
    axes[0].set_ylabel('P(minigame)'); axes[0].set_ylim(0, 1.05)
    axes[0].set_title(f'GT vs {lbl} — {GAME_PREFIX}', fontweight='bold')
    axes[0].legend(fontsize=7, ncol=len(MINIGAMES)*2, loc='upper right')
    axes[0].grid(alpha=0.2)
    divergence = np.abs(gt_posts - trk_posts).sum(axis=1) / 2
    axes[1].fill_between(x, divergence, alpha=0.4, color='crimson')
    axes[1].plot(x, divergence, color='crimson', lw=1)
    axes[1].set_ylabel('L1 divergence'); axes[1].set_ylim(0, 1)
    axes[1].set_xlabel('Frame (global)')
    axes[1].set_title('Posterior Divergence (0=identical, 1=opposite)', fontweight='bold')
    for _, start, _ in chunk_boundaries:
        axes[1].axvline(start, color='gray', lw=0.5, alpha=0.4)
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    _slug = lbl.replace(' ', '_').replace('+', 'x')
    _out  = OUT_DIR / f'{GAME_PREFIX}_{_slug}_divergence.png'
    plt.savefig(_out, dpi=150, bbox_inches='tight'); plt.show()
    res['mean_divergence'] = float(divergence.mean())
    res['max_divergence']  = float(divergence.max())
    res['max_div_frame']   = int(divergence.argmax())
    print(f'{lbl}: mean_div={divergence.mean():.3f}  max_div={divergence.max():.3f}')


## 7 — Stat Count Comparison Table

Per-class count of unique track IDs seen across the entire game,
with GT vs tracker difference.

In [ ]:
# ── Per-run stat count table ──────────────────────────────────────────────
for lbl, res in all_run_results.items():
    _df = pd.DataFrame({
        'GT counts':      gt_counts,
        'Tracker counts': res['counts'],
        'Difference':     res['counts'] - gt_counts,
        '% Error':        np.where(gt_counts > 0,
                                   (res['counts'] - gt_counts) / gt_counts * 100, np.nan),
    }, index=CLASSES)
    print(f'\n── {lbl} — Stat Count Comparison ──')
    display(_df.style.format({'% Error': '{:.1f}%'})
                     .background_gradient(subset=['Difference'], cmap='RdYlGn_r'))

# ── Summary table ─────────────────────────────────────────────────────────
summary_rows = []
for lbl, res in all_run_results.items():
    final_post_trk = res['posts'][-1]
    final_post_gt  = gt_posts[-1]
    summary_rows.append({
        'Run':             lbl,
        'Elapsed (s)':     round(res['elapsed'], 1),
        'Elapsed (min)':   round(res['elapsed'] / 60, 2),
        'Mean Divergence': round(res.get('mean_divergence', float('nan')), 4),
        'Max Divergence':  round(res.get('max_divergence',  float('nan')), 4),
        'Unique Tracks':   int(res['trk_df']['track_id'].nunique()),
        'Total Counts':    int(res['counts'].sum()),
        'GT Counts':       int(gt_counts.sum()),
        'Count % Error':   round(float(np.where(gt_counts.sum() > 0,
                               (res['counts'].sum() - gt_counts.sum()) / gt_counts.sum() * 100,
                               float('nan'))), 2),
    })

df_summary = pd.DataFrame(summary_rows).set_index('Run')
print('\n' + '='*60)
print(f'SUMMARY — {GAME_PREFIX}')
print('='*60)
display(df_summary.style
    .format({'Elapsed (s)': '{:.1f}', 'Elapsed (min)': '{:.2f}',
             'Mean Divergence': '{:.4f}', 'Max Divergence': '{:.4f}',
             'Count % Error': '{:+.1f}%'})
    .background_gradient(subset=['Mean Divergence'], cmap='YlOrRd')
    .background_gradient(subset=['Elapsed (s)'],     cmap='Blues')
    .background_gradient(subset=['Count % Error'],   cmap='RdYlGn_r'))

df_summary.to_csv(OUT_DIR / f'{GAME_PREFIX}_run_summary.csv')
print(f'\nSaved: {OUT_DIR}/{GAME_PREFIX}_run_summary.csv')
